# Federated Random Forest — Union of Forests

**Coordinator notebook in the federated architecture.**

The four `RandomForest_<STUDY>` notebooks each fit a forest on one study alone.
This notebook combines them as a **Union-of-Forests**: every institution keeps
its own forest, a subject is scored by **every** forest, and the predictions
are averaged. Trees and subject rows never move between institutions — only the
per-subject predictions are averaged at inference.

Random forest has only one natural federated aggregation (the union), so here
we compare **Union-of-Forests** against the **Solo** baseline (each study's own
forest on its own held-out rows). We also aggregate the per-study feature-
importance vectors into a federated ranking, and show the additive-benefit
cohort curves.

## Cohorts
- Panel A — 4 studies, 9 features.
- Panel B — 3 studies, 12 features.
- Panel B excluding SDY1737 — the deck's headline cohort.


## 1. Setup

In [ ]:
from __future__ import annotations
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from scipy.stats import f as fdist, ncf

import oadr_data as od

RNG_SEED = 42
N_TREES = 200
N_FOLDS = 5
ALPHA_STAT = 0.05
np.random.seed(RNG_SEED)
(REPO / "results").mkdir(exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)

PANEL_A_FEATS = ['MIAA', 'GAD65', 'IA2IC', 'ICA', 'ZNT8', '8-12', '13-17', '>18', 'Sex']
PANEL_B_FEATS = ['Sex', 'age_years', 'disease_duration_years', 'bmi', 'height_cm', 'weight_kg', 'GAD65', 'IA2IC', 'MIAA', 'ZNT8', 'ICA', 'received_active_treatment']
PANEL_A_STUDIES = ["SDY524", "SDY569", "SDY797", "SDY1737"]
PANEL_B_STUDIES = ["SDY524", "SDY569", "SDY1737"]
print("Repo root:", REPO)


## 2. Aggregate the per-study importance vectors

Read each `vectors/<STUDY>_rf_panel<X>_importance.csv` and report, per feature,
the per-study importances and their median across studies — a federated view of
what the forests relied on.


In [ ]:
def importance_table(panel, studies, feats):
    cols = {}
    for s in studies:
        p = REPO / "vectors" / f"{s}_rf_panel{panel}_importance.csv"
        if not p.exists():
            continue
        cols[s] = pd.read_csv(p).set_index("feature")["importance"]
    tbl = pd.DataFrame(cols).reindex(feats)
    tbl["median"] = tbl.median(axis=1)
    return tbl.sort_values("median", ascending=False)

print("=== Panel A importance (median across studies) ===")
impA = importance_table("A", PANEL_A_STUDIES, PANEL_A_FEATS)
print(impA.round(3).to_string())
impA.to_csv("vectors/federated_rf_panelA_importance.csv")
print()
print("=== Panel B importance (median across studies) ===")
impB = importance_table("B", PANEL_B_STUDIES, PANEL_B_FEATS)
print(impB.round(3).to_string())
impB.to_csv("vectors/federated_rf_panelB_importance.csv")


## 3. Honest federated CV — Union-of-Forests vs Solo

Each study splits its **own** rows. In each fold every study fits a forest on
its training rows (MinMaxScaler per study). For the **Union**, every study's
held-out subject is scored by **all** the forests (each forest scales the
subject with its own scaler) and the predictions are averaged. For **Solo**,
each study's subject is scored only by its own forest. Each study reports
summary scalars (RSS, Σy, Σy², n); the coordinator combines them. No subject
rows, and no trees, are pooled.


In [ ]:
def calc_power(n, k, f2, alpha=ALPHA_STAT):
    if n <= k + 1 or f2 <= 0:
        return float("nan")
    F_crit = fdist.ppf(1 - alpha, k, n - k - 1)
    return float(1 - ncf.cdf(F_crit, k, n - k - 1, f2 * n))

def f2_from_r2(r2):
    return r2 / (1 - r2) if 0 < r2 < 1 else 0.0

def study_summary(y, pred):
    m = ~np.isnan(pred); yy, pp = y[m], pred[m]
    return {"rss": float(np.sum((yy - pp) ** 2)), "sum_y": float(np.sum(yy)),
            "sum_y2": float(np.sum(yy ** 2)), "n": int(m.sum())}

def score_from_summaries(summaries, k):
    N = sum(d["n"] for d in summaries); RSS = sum(d["rss"] for d in summaries)
    SY = sum(d["sum_y"] for d in summaries); SY2 = sum(d["sum_y2"] for d in summaries)
    gmean = SY / N; TSS = SY2 - N * gmean ** 2
    mse = RSS / N; r2 = 1 - RSS / TSS if TSS > 0 else float("nan")
    return mse, r2, N, calc_power(N, k, f2_from_r2(r2))

def load_study(panel, study, feats):
    if panel == "A":
        a = od.load_panel_a(study)
        return a[feats].values.astype(float), a[od.PANEL_A_TARGET].values.astype(float)
    b = od.load_panel_b(study)
    for col in ("bmi", "height_cm", "weight_kg"):
        b[col] = b[col].fillna(b[col].median())
    bad = b["height_cm"] <= 0
    b.loc[bad, "height_cm"] = np.sqrt(b.loc[bad, "weight_kg"] / b.loc[bad, "bmi"]) * 100
    X, y, _ = od.panel_b_design_matrix(b)
    return X.reindex(columns=feats).values.astype(float), y.values.astype(float)

def federated_cv(panel, studies, feats, n_splits=N_FOLDS, seed=RNG_SEED):
    data = {s: load_study(panel, s, feats) for s in studies}
    folds = {s: list(KFold(n_splits=min(n_splits, max(2, len(data[s][1]) // 2)),
                           shuffle=True, random_state=seed).split(data[s][0]))
             for s in studies}
    nf = min(len(folds[s]) for s in studies)
    oof = {r: {s: np.full(len(data[s][1]), np.nan) for s in studies} for r in ["Union", "Solo"]}
    for k in range(nf):
        forests, scalers = {}, {}
        for s in studies:
            X, y = data[s]; tr, te = folds[s][k]
            sc = MinMaxScaler().fit(X[tr]); scalers[s] = sc
            rf = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=2,
                                       n_jobs=1, random_state=seed).fit(sc.transform(X[tr]), y[tr])
            forests[s] = rf
            oof["Solo"][s][te] = rf.predict(sc.transform(X[te]))
        for s in studies:
            X, y = data[s]; tr, te = folds[s][k]
            preds = [forests[t].predict(scalers[t].transform(X[te])) for t in studies]
            oof["Union"][s][te] = np.mean(preds, axis=0)
    rows = []
    for r in ["Union", "Solo"]:
        summ = [study_summary(data[s][1], oof[r][s]) for s in studies]
        mse, r2, n, pwr = score_from_summaries(summ, len(feats))
        rows.append({"panel": panel, "rule": r, "N": n, "k": len(feats),
                     "MSE": mse, "R2": r2, "achieved_power": pwr})
    return pd.DataFrame(rows)

COHORTS = [
    ("A", PANEL_A_STUDIES, PANEL_A_FEATS, "Panel A (4 studies)"),
    ("B", PANEL_B_STUDIES, PANEL_B_FEATS, "Panel B (3 studies)"),
    ("B", ["SDY524", "SDY569"], PANEL_B_FEATS, "Panel B excl. SDY1737"),
]
perf = []
for panel, studies, feats, label in COHORTS:
    df = federated_cv(panel, studies, feats); df["cohort"] = label
    perf.append(df)
perf = pd.concat(perf, ignore_index=True)
print(perf[["cohort", "rule", "N", "MSE", "R2", "achieved_power"]].to_string(index=False))
perf.to_csv("results/federated_rf_performance.csv", index=False)


## 4. Additive benefit — cohort growth (smallest N first)

Add the studies one at a time, smallest N first; re-run the Union-of-Forests
CV at each step.

- Panel A: SDY569 (10) → +SDY1737 (16) → +SDY797 (49) → +SDY524 (75)
- Panel B: SDY569 (10) → +SDY1737 (16) → +SDY524 (72)


In [ ]:
GROWTH_ORDER = {"A": ["SDY569", "SDY1737", "SDY797", "SDY524"],
                "B": ["SDY569", "SDY1737", "SDY524"]}

def cohort_growth(panel, feats):
    order = GROWTH_ORDER[panel]; rec = []
    for i in range(1, len(order) + 1):
        sub = order[:i]
        res = federated_cv(panel, sub, feats).set_index("rule")
        n = int(res.loc["Union", "N"])
        for rule in ["Union", "Solo"]:
            rec.append({"panel": panel, "n_studies": i, "cohort_N": n, "added": sub[-1],
                        "rule": rule, "R2": res.loc[rule, "R2"],
                        "achieved_power": res.loc[rule, "achieved_power"]})
    return pd.DataFrame(rec)

growthA = cohort_growth("A", PANEL_A_FEATS)
growthB = cohort_growth("B", PANEL_B_FEATS)
growth = pd.concat([growthA, growthB], ignore_index=True)
growth.to_csv("results/federated_rf_cohort_growth.csv", index=False)
print(growth.to_string(index=False))


## 5. Graphics


In [ ]:
# Federated importance ranking (median across studies), per panel
def plot_importance(imp, panel):
    d = imp["median"].dropna().sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(d) + 1))
    ax.barh(d.index, d.values, color="#1f77b4")
    ax.set_xlabel("Median impurity-decrease importance (across studies)")
    ax.set_title(f"Federated RF importance — Panel {panel}", fontweight="bold")
    ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    fig.savefig(f"figures/federated_rf_importance_panel{panel}.pdf", dpi=300)
    fig.savefig(f"figures/federated_rf_importance_panel{panel}.png", dpi=220)
    plt.show()

plot_importance(impA, "A")
plot_importance(impB, "B")


In [ ]:
def plot_growth(panel, g):
    order = GROWTH_ORDER[panel]
    labels = [f"{i}: +{order[i-1]}" for i in range(1, len(order) + 1)]
    colors = {"Union": "#1f77b4", "Solo": "0.6"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
    for rule, col in colors.items():
        d = g[g.rule == rule].sort_values("n_studies")
        axes[0].plot(d.n_studies, d.R2, marker="o", color=col, label=rule)
        axes[1].plot(d.n_studies, d.achieved_power.fillna(0.0), marker="o", color=col, label=rule)
    axes[0].axhline(0, color="k", lw=0.6); axes[0].set_ylabel("R² (federated CV)")
    axes[1].axhline(0.80, color="r", ls="--", lw=1, label="80% power"); axes[1].set_ylim(0, 1.0)
    axes[1].set_ylabel("Achieved power")
    for ax in axes:
        ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_xlabel("cohort (smallest first)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    axes[0].set_title(f"R² grows with cohort — Panel {panel}", fontweight="bold")
    axes[1].set_title(f"Power grows with cohort — Panel {panel}", fontweight="bold")
    fig.savefig(f"figures/federated_rf_cohort_growth_panel{panel}.pdf", dpi=300)
    fig.savefig(f"figures/federated_rf_cohort_growth_panel{panel}.png", dpi=220)
    plt.show()

plot_growth("A", growthA)
plot_growth("B", growthB)


## 6. Outputs

CSVs:
- `vectors/federated_rf_panel{A,B}_importance.csv` — per-study + median importance.
- `results/federated_rf_performance.csv` — Union vs Solo MSE / R² / power per cohort.
- `results/federated_rf_cohort_growth.csv` — R² and power at each cohort size.

Figures:
- `federated_rf_importance_panel{A,B}` — federated importance ranking.
- `federated_rf_cohort_growth_panel{A,B}` — additive-benefit curves.

This completes the three interpretable methods (Ridge, LASSO, Random Forest),
each as per-study notebooks plus a federated coordinator with its own
aggregation rule.
